## Topic: Complete Implementation of Wikipedia_retriever with FAISS Database

- Wikipedia → Split → FAISS similarity retriever


In [ ]:
"""
- 1. First install FAISS 
    - pip install faiss-cpu

    
"""

### 1. Import Necessary Library

In [2]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


### 2. Initialize the WikipediaRetriever
- All Retrievers are Runnable

In [8]:
from langchain_community.retrievers import WikipediaRetriever

wiki = WikipediaRetriever(
    top_k_results=3,
    doc_content_chars_max=1000
)

docs = wiki.invoke("Generative artificial intelligence")

print(f"Number of documents: {len(docs)}")

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Document {i} ---")
    print("Title:", doc.metadata.get("title"))
    print("Content:")
    print(doc.page_content[:500])

Number of documents: 3

--- Document 1 ---
Title: Generative AI
Content:
Generative artificial intelligence (GenAI) is a subfield of artificial intelligence (AI) that uses generative models to generate text, images, videos, audio, software code or other forms of data. These models learn the underlying patterns and structures of their training data, and use them to generate new data in response to input, which often takes the form of natural language prompts.
The prevalence of generative AI tools has increased significantly since the AI boom in the 2020s. This boom wa

--- Document 2 ---
Title: Generative artificial intelligence dependency
Content:
Generative artificial intelligence dependency (GAID) is an emerging artificial intelligence disorder where users have a compulsive reliance on AI tools for tasks involving creativity, critical thinking, and emotional support. The excessive offloading of individual agency to AI may stunt psychosocial development and critical thinking.


== See

### 3. Split the Wikipedia Response

In [9]:
# 2) Split
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_documents(docs)

### 4. Embedding and store the FAISS Database

In [10]:
# 3) Embed + store in FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

faiss_db = FAISS.from_documents(chunks, embeddings)

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1386.27it/s]


### 5. Retrievers the Data from FAISS Database

In [11]:
# 4) Local similarity retriever (fast, reusable)
faiss_retriever = faiss_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)


### 6. Get the Response

In [14]:

docs = faiss_retriever.invoke("What is Generative artificial intelligence?")

for i, d in enumerate(docs):
    print(f"Response: {i + 1}:\n{d.page_content[:200]}", "\n")

Response: 1:
Generative artificial intelligence (GenAI) is a subfield of artificial intelligence (AI) that uses generative models to generate text, images, videos, audio, software code or other forms of data. 

Response: 2:
Companies in a variety of sectors have used generative AI, including those in 

Response: 3:
Generative artificial intelligence dependency (GAID) is an emerging artificial intelligence disorder where users have a compulsive reliance on AI tools for tasks involving creativity, critical 

Response: 4:
The prevalence of generative AI tools has increased significantly since the AI boom in the 2020s. This boom was made possible by improvements in deep neural networks, particularly large language 



In [ ]:

# at a glance
from langchain_community.retrievers import WikipediaRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# object of WikipediaRetriever
wiki = WikipediaRetriever(
    top_k_results=3,
    lang = "en", 
    doc_content_chars_max=1000
    )

# 1) Fetch Wikipedia documents
raw_docs = wiki.invoke("Transformer (machine learning)")

# 2) Split
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

# 3) Embed + store in FAISS
embeddings = OpenAIEmbeddings()
faiss_db = FAISS.from_documents(chunks, embeddings)

# 4) Local similarity retriever (fast, reusable)
faiss_retriever = faiss_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

docs = faiss_retriever.invoke("What is multi-head attention?")
for d in docs:
    print(d.page_content[:200], "\n")